In [1]:
!pip -q install sentence-transformers pypdf gradio requests

In [8]:

import os
import requests
import gradio as gr
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google.colab import userdata

# -------------------------------
# 1. LOAD EMBEDDING MODEL
# -------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# -------------------------------
# 2. GLOBAL DOCUMENT STORAGE
# -------------------------------

document_chunks = []
document_embeddings = []

# -------------------------------
# 3. READ PDF
# -------------------------------

def process_pdf(pdf_file):

    global document_chunks, document_embeddings

    if pdf_file is None:
        return "Please upload a PDF."

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    if not text.strip():
        return "No readable text found in this PDF."

    # Split text into chunks
    chunk_size = 500
    overlap = 100

    document_chunks = []

    for i in range(0, len(text), chunk_size - overlap):

        chunk = text[i:i + chunk_size]

        if chunk.strip():
            document_chunks.append(chunk)

    # Convert chunks into embeddings
    document_embeddings = embedding_model.encode(
        document_chunks,
        normalize_embeddings=True
    )

    return (
        f"PDF processed successfully!\n"
        f"Total chunks: {len(document_chunks)}"
    )

# -------------------------------
# 4. RETRIEVAL
# -------------------------------

def retrieve_context(question):

    if not document_chunks:
        return None

    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    similarities = np.dot(
        document_embeddings,
        question_embedding
    )

    top_indices = np.argsort(similarities)[-3:][::-1]

    context = "\n\n".join(
        [document_chunks[i] for i in top_indices]
    )

    return context

# -------------------------------
# 5. OLLAMA CLOUD LLM
# -------------------------------

def generate_answer(question, context):

    api_key = userdata.get("OLLAMA_API_KEY")

    if not api_key:
        return "Please add your OLLAMA_API_KEY in Colab Secrets."

    url = "https://ollama.com/api/chat"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    prompt = f"""
    You are a helpful PDF assistant.

    Answer the question using the supplied document context.

    If the answer is not available in the context,
    clearly say that it was not found in the PDF.

    DOCUMENT CONTEXT:
    {context}

    QUESTION:
    {question}
    """

    payload = {
        "model": "gemma3",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    return response.json()["message"]["content"]

# -------------------------------
# 6. CHAT FUNCTION
# -------------------------------

def chat(question, history):

    if not question.strip():
        return "Please enter a question."

    context = retrieve_context(question)

    if context is None:
        return "Please upload and process a PDF first."

    try:
        answer = generate_answer(question, context)
        return answer

    except Exception as e:
        return f"Error connecting to Ollama: {e}"

# -------------------------------
# 7. GRADIO INTERFACE
# -------------------------------

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # AI PDF Assistant
        ### Retrieval-Augmented Generation using RAG + Ollama

        Upload your PDF and ask questions about its content.
        """
    )

    pdf_input = gr.File(
        label="Upload your PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button("Process PDF")

    status = gr.Textbox(
        label="Processing Status"
    )

    process_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    chatbot = gr.ChatInterface(
        fn=chat,
        title="Ask Your PDF",
        chatbot=gr.Chatbot()
    )

demo.launch(share=True)



Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_4744/3249422853.py:179: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://965752391401d81c4e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:

import requests
from google.colab import userdata

# Get your saved API key
api_key = userdata.get("OLLAMA_API_KEY")

# Check API key
if not api_key:
    print("API key not found. Check Colab Secrets.")
else:
    print("API key found successfully!")

    # Test Ollama connection
    url = "https://ollama.com/api/chat"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "gemma4:31b",
        "messages": [
            {
                "role": "user",
                "content": "Say hello in one sentence."
            }
        ],
        "stream": False
    }

    try:
        response = requests.post(
            url,
            headers=headers,
            json=data,
            timeout=180
        )

        print("Status code:", response.status_code)

        if response.status_code == 200:
            print("Ollama connection successful!")
            print("AI Response:")
            print(response.json()["message"]["content"])
        else:
            print("Ollama connection failed.")
            print("Error details:", response.text)

    except Exception as e:
        print("Connection error:", e)



API key found successfully!
Status code: 402
Ollama connection failed.
Error details: {"error":"Add usage credits or subscribe and save up to 66%: https://ollama.com/plans (ref: cbd805a8-eea8-4a90-8bbc-11088ba522f1)"}

